# Baselines de ML estándar y chequeo de aprendibilidad

Este notebook entrena clasificadores estándar (LogisticRegression, DecisionTree, RandomForest) sobre el **mismo split y preprocesado** que LAMDA, usando el módulo `lamda.experiments.baselines`. Sitúa a LAMDA respecto a modelos habituales sin mitigar y diagnostica la aprendibilidad de cada dataset frente al baseline de clase mayoritaria.

In [1]:
import pandas as pd
from lamda.experiments import baselines
from lamda.experiments import runner
from lamda.experiments import config as cfg
from lamda.fairness.disparity import FairnessThresholds

pd.set_option("display.float_format", "{:.4f}".format)
THRESHOLDS = FairnessThresholds(spd=0.1, eod=0.1, di=0.8)

# Las celdas finales del notebook recogen la exploración previa que fijó la
# exigencia y el operador y que descartó varios conjuntos candidatos. Sus
# conclusiones ya están incorporadas al protocolo, de modo que no se repiten en
# una ejecución normal. Poner a True para reproducirla.
EJECUTAR_EXPLORACION = False


## Baselines por dataset + comparación con LAMDA clásico

Para cada dataset se muestran los estándar y la clase mayoritaria, y se contrasta con la accuracy del LAMDA clásico (vía `run_detection`). La columna `supera_trivial` marca si el mejor estándar bate a predecir siempre la clase mayoritaria; si LAMDA no lo hace y los estándar sí, el problema es de configuración de LAMDA, no del dataset.

In [2]:
# Atributo sensible por conjunto. Solo OULAD se aparta del que trae su
# configuración: emplea la discapacidad, porque con el género su disparidad de
# partida es nula y no deja margen sobre el que medir ni corregir.
MODO_SENSIBLE_POR_DATASET = {"oulad": "disability"}

def modo_sensible(dataset):
    """Atributo sensible de un conjunto. None emplea el de la configuración."""
    return MODO_SENSIBLE_POR_DATASET.get(dataset)

resumen = []
for name in cfg.all_datasets():
    modo = modo_sensible(name)
    res = baselines.run_baselines(dataset_name=name, sensitive_mode=modo,
                                  thresholds=THRESHOLDS)
    baselines.persist_baselines(name, res)
    maj = res["majority_accuracy"]

    # LAMDA clásico sobre la misma partición y el mismo atributo sensible.
    det = runner.run_detection(dataset_name=name, sensitive_mode=modo,
                               thresholds=THRESHOLDS)
    fila_lamda = det.detection_table.iloc[0]
    acc_lamda = float(fila_lamda["Accuracy"])
    f1_lamda = float(fila_lamda["F1_macro"])

    tabla = res["table"]
    etiqueta = modo if modo is not None else cfg.DATASETS[name].sensitive_mode
    print(f"\n=== {cfg.DATASETS[name].display_name} | atributo={etiqueta} | "
          f"mayoritaria={maj:.4f} | LAMDA={acc_lamda:.4f} ===")
    cols = [c for c in ["Accuracy", "F1_macro", "mean_disparity", "mean_abs_spd",
                        "mean_abs_eod", "min_di"] if c in tabla.columns]
    display(tabla[cols])

    sin_trivial = tabla.drop(index="Clase mayoritaria", errors="ignore")
    mejor = float(sin_trivial["Accuracy"].max())
    # F1 macro del predictor de clase mayoritaria. Es la magnitud sobre la que
    # se evalúa la condición previa de aprendibilidad del criterio de
    # calibración, que bajo desbalance distingue mejor que la exactitud entre un
    # clasificador que discrimina entre clases y uno que no.
    f1_maj = (float(tabla.loc["Clase mayoritaria", "F1_macro"])
              if "Clase mayoritaria" in tabla.index else float("nan"))
    resumen.append({
        "Dataset": cfg.DATASETS[name].display_name,
        "Atributo sensible": etiqueta,
        "mayoritaria": maj,
        "LAMDA": acc_lamda,
        "mejor_estandar": mejor,
        "F1 macro LAMDA": f1_lamda,
        "F1 macro mayoritaria": f1_maj,
        "LAMDA_supera_trivial": acc_lamda > maj,
        "LAMDA_supera_trivial_F1": f1_lamda > f1_maj,
        "estandar_supera_trivial": mejor > maj,
    })



=== Adult | atributo=sex | mayoritaria=0.7510 | LAMDA=0.7654 ===


,Accuracy,F1_macro,mean_disparity,mean_abs_spd,mean_abs_eod,min_di
Modelo,,,,,,
LogisticRegression,0.8437,0.7765,0.3439,0.1810,0.1040,0.3072
DecisionTree,0.8134,0.7474,0.2899,0.1818,0.0668,0.3917
RandomForest,0.8470,0.7871,0.3166,0.1848,0.0644,0.3409
Clase mayoritaria,0.7510,0.4289,0.0000,0.0000,0.0000,1.0000



=== Dutch Census | atributo=sex | mayoritaria=0.5239 | LAMDA=0.7282 ===


,Accuracy,F1_macro,mean_disparity,mean_abs_spd,mean_abs_eod,min_di
Modelo,,,,,,
LogisticRegression,0.8361,0.8354,0.5460,0.3194,0.1237,0.4824
DecisionTree,0.8196,0.8187,0.5648,0.3251,0.1413,0.4747
RandomForest,0.8301,0.8296,0.6048,0.3403,0.1501,0.4695
Clase mayoritaria,0.5239,0.3438,0.0000,0.0000,0.0000,1.0000



=== COMPAS | atributo=race | mayoritaria=0.5294 | LAMDA=0.6117 ===


...

,Accuracy,F1_macro,mean_disparity,mean_abs_spd,mean_abs_eod,min_di
Modelo,,,,,,
LogisticRegression,0.4435,0.3064,0.1583,0.1333,0.1454,0.0000
DecisionTree,0.3419,0.2898,0.0000,0.0319,0.0316,0.7013
RandomForest,0.3913,0.3134,0.0576,0.0928,0.0912,0.6072
Clase mayoritaria,0.3757,0.1366,0.0000,0.0000,0.0000,1.0000



=== Communities and Crime | atributo=black | mayoritaria=0.3409 | LAMDA=0.5564 ===


,Accuracy,F1_macro,mean_disparity,mean_abs_spd,mean_abs_eod,min_di
Modelo,,,,,,
LogisticRegression,0.6942,0.6867,0.3541,0.3107,0.2158,0.1854
DecisionTree,0.6040,0.6009,0.2104,0.2472,0.1318,0.2969
RandomForest,0.6917,0.6831,0.3523,0.3080,0.2074,0.2048
Clase mayoritaria,0.3409,0.1695,0.0000,0.0000,0.0000,1.0000



=== Student Performance | atributo=sex | mayoritaria=0.4615 | LAMDA=0.5077 ===


,Accuracy,F1_macro,mean_disparity,mean_abs_spd,mean_abs_eod,min_di
Modelo,,,,,,
LogisticRegression,0.5692,0.5143,0.1632,0.1413,0.1795,0.5056
DecisionTree,0.5077,0.4903,0.0678,0.0302,0.1344,0.8623
RandomForest,0.6308,0.5512,0.1025,0.0968,0.1779,0.7000
Clase mayoritaria,0.4615,0.2105,0.0000,0.0000,0.0000,1.0000


In [3]:
# Resumen de aprendibilidad de los seis conjuntos, en exactitud y en F1 macro.
resumen_aprendibilidad = pd.DataFrame(resumen).set_index("Dataset")
resumen_aprendibilidad.to_csv(
    __import__("os").path.join(cfg.RESULTS_DIR, "aprendibilidad.csv"))
resumen_aprendibilidad


,Atributo sensible,mayoritaria,LAMDA,mejor_estandar,F1 macro LAMDA,F1 macro mayoritaria,LAMDA_supera_trivial,LAMDA_supera_trivial_F1,estandar_supera_trivial
Dataset,,,,,,,,,
Adult,sex,0.7510,0.7654,0.8470,0.7127,0.4289,True,True,True
Dutch Census,sex,0.5239,0.7282,0.8361,0.7282,0.3438,True,True,True
COMPAS,race,0.5294,0.6117,0.6799,0.5985,0.3461,True,True,True
OULAD,disability,0.3757,0.2871,0.4435,0.2707,0.1366,False,True,True
Communities and Crime,black,0.3409,0.5564,0.6942,0.5530,0.1695,True,True,True
Student Performance,sex,0.4615,0.5077,0.6308,0.4721,0.2105,True,True,True


## Exploración previa (desactivada por defecto)

Las celdas siguientes recogen la exploración que fijó la exigencia y el operador
de agregación y la comprobación de aprendibilidad de varios conjuntos candidatos
que no entraron en el panel final. Se conservan como registro y no se ejecutan
salvo que se ponga `EJECUTAR_EXPLORACION = True`.


In [4]:
# Exploración de la exigencia y del operador de agregación. Fijó los valores
# adoptados en el protocolo. Se restringe a OULAD, que es el conjunto del panel
# cuyo rendimiento queda por debajo del predictor trivial; la versión original
# incluía además un conjunto candidato que no forma parte del panel final.
if EJECUTAR_EXPLORACION:
    import numpy as np
    from collections import Counter
    TH = FairnessThresholds(spd=0.1, eod=0.1, di=0.8)

    alphas = [0.5, 0.6, 0.65, 0.7, 0.8, 0.9]
    operators = ["minmax", "product", "lukasiewicz"]

    for name in ["oulad"]:
        modo = modo_sensible(name)
        Xtr, Xte, ytr, yte, *_ = runner.load_and_split(name, sensitive_mode=modo)
        maj_cls = Counter(np.asarray(ytr).tolist()).most_common(1)[0][0]
        trivial = float((np.asarray(yte) == maj_cls).mean())
        rows = []
        for op in operators:
            for a in alphas:
                det = runner.run_detection(dataset_name=name, sensitive_mode=modo,
                                           alpha=a, operator=op, thresholds=TH)
                rows.append({"operator": op, "alpha": a,
                             "acc": float(det.detection_table.iloc[0]["Accuracy"])})
        piv = pd.DataFrame(rows).pivot(index="alpha", columns="operator", values="acc")
        print(f"\n=== {name} | trivial = {trivial:.4f} ===")
        display(piv)
else:
    print("Exploración desactivada (EJECUTAR_EXPLORACION = False).")


Exploración desactivada (EJECUTAR_EXPLORACION = False).


In [5]:
# Comprobación puntual sobre Dutch Census durante su incorporación al panel.
if EJECUTAR_EXPLORACION:
    import numpy as np
    from collections import Counter
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import MinMaxScaler
    from sklearn.ensemble import RandomForestClassifier
    from lamda.models.lamda_classifier import LamdaClassifier
    from lamda.data.load_dutch import load_dutch_dataset

    X, y, s, f = load_dutch_dataset(sensitive_mode="sex")
    X = np.asarray(X, dtype=float); y = np.asarray(y)
    print("Dutch:", X.shape, "clases", np.unique(y, return_counts=True),
          "sexo", np.unique(s, return_counts=True))
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2,
                                          random_state=42, stratify=y)
    maj = Counter(ytr.tolist()).most_common(1)[0][0]
    trivial = (yte == maj).mean()
    lam = LamdaClassifier(alpha=0.65, operator="minmax").fit(Xtr, ytr)
    acc_lam = (lam.predict(Xte) == yte).mean()
    sc = MinMaxScaler().fit(Xtr)
    rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(
        sc.transform(Xtr), ytr)
    acc_rf = (rf.predict(sc.transform(Xte)) == yte).mean()
    print(f"trivial={trivial:.4f} LAMDA={acc_lam:.4f} RF={acc_rf:.4f} "
          f"LAMDA>trivial={acc_lam>trivial}")
else:
    print("Exploración desactivada (EJECUTAR_EXPLORACION = False).")


Exploración desactivada (EJECUTAR_EXPLORACION = False).
